# Dataset processing

Input data: 
- ParlaMint dataset (handle), where the data is merged together into a master preprocessed dataset, to be used for all of the analysis. Specifically, we include the following data:

Sample dataset (for replicability purposes:) in the [Sample](../Sample/ParlaMint-SI5.1_Sample/):
- [Sample.txt](../Sample/ParlaMint-SI5.1_Sample/Sample.txt/) (text files, to acquire the texts)
- [Sample.conllu](../Sample/ParlaMint-SI5.1_Sample/Sample.conllu/) (metadata and sentiment metadata files)
- [CHES dataset for Slovenian data](../Sample/CHES-SI.tsv)

Sample timeframe:
This sample covers Nov 2019 – Apr 2020, a key period in Slovenian politics that overlaps with the start of COVID-19. It’s included both to demonstrate the analysis workflow and to give a snapshot of party and speech trends during this timeframe—useful context when interpreting results from the full dataset.


Full dataset available for download on the CLARIN.SI repository: 
> Erjavec, Tomaž; et al., 2025, **Multilingual comparable corpora of parliamentary debates ParlaMint 5.0**, Slovenian language resource repository CLARIN.SI, ISSN 2820-4042, 
http://hdl.handle.net/11356/2004.

Full CHES dataset is available for download through Chapel Hill Expert Survey website:
> Jolly, Seth, Ryan Bakker, Liesbet Hooghe, Gary Marks, Jonathan Polk, Jan Rovny, Marco Steenbergen, and Milada Anna Vachudova. 2022. **Chapel Hill Expert Survey Trend File, 1999-2019.** Electoral Studies 75 (February). https://doi.org/10.1016/j.electstud.2021.102420

The already processed CHES-SI dataset (with aligned ParlaMint party IDs) is available through [ParlaMint GitHub site](https://github.com/clarin-eric/ParlaMint/tree/main/Build/Sources-TSV/ParlaMint-SI/OrientationCHES-SI.edited.tsv).


Output:
- Pre-processed and merged master dataset [ParlaMint-SI_5.1_master.tsv](../Sample/Datasets/Sample_master.tsv)

## Preprocessing ParlaMint-SI corpus

In [1]:
import pandas as pd
import os
import glob

In [2]:
#Sample datasets, replace with actual ParlaMint dataset from CLARIN.SI repository entry
#text_dir = "../Sample/ParlaMint-SI5.1_Sample/Sample.txt"
#conllu_dir = "../Sample/ParlaMint-SI5.1_Sample/Sample.conllu"


In [3]:
text_dir = "../ParlaMint-SI5.1/ParlaMint-SI.txt"
conllu_dir = "../ParlaMint-SI5.1/ParlaMint-SI.conllu"

In [4]:
# Text files to extrac ID and text
text_files = glob.glob(os.path.join(text_dir, '*', '*.txt'))

#Session metadata
text_meta = glob.glob(os.path.join(text_dir, '*', '*-meta-en.tsv'))

#Sentiment-related metadata
sent_files = glob.glob(os.path.join(conllu_dir, '**', '*-ana-meta-en.tsv'))

print(text_meta)

['../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-03-28-SDZ6-Redna-12-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-04-02-SDZ6-Redna-12-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-06-18-SDZ6-Redna-15-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-11-19-SDZ6-Redna-19-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-07-09-SDZ6-Redna-16-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-05-24-SDZ6-Izredna-32-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-09-24-SDZ6-Redna-17-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-02-01-SDZ6-Redna-10-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-12-12-SDZ6-Izredna-51-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-SI_2013-10-29-SDZ6-Izredna-46-meta-en.tsv', '../ParlaMint-SI5.1/ParlaMint-SI.txt/2013/ParlaMint-S

In [5]:
id = []
texts = []
#Processing text files
for file in text_files:
    with open(file, 'r', encoding="utf-8") as infile:
        for line in infile:
            if '\t' in line:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    id.append(parts[0])
                    texts.append(parts[1])


df_texts = pd.DataFrame({'ID': id, 'Text':texts})
print("df_text shape: ", df_texts.shape)

df_texts.head()

df_text shape:  (311354, 2)


,ID,Text
0,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u1,Spoštovane kolegice poslanke in kolegi poslanc...
1,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u2,Hvala lepa za besedo. Spoštovana predsednica V...
2,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u3,Hvala tudi vam. Dajem besedo predsednici Vlade...
3,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u4,"Spoštovane poslanke in poslanci, lep pozdrav p..."
4,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u5,Hvala za odgovor. Gospod Prevc bo predstavil z...


In [6]:
df_meta = pd.DataFrame()
chunk_size = 1000 

for file in text_meta:
    df = pd.read_csv(file, sep='\t', encoding='utf-8')

    
    if 'Body' in df.columns:
        chunked_rows = []
        for _, row in df.iterrows():
            body_text = str(row['Body'])
            text_id = str(row.get('Text_ID', 'unknown'))

            # Split long text into smaller chunks
            for i in range(0, len(body_text), chunk_size):
                chunk = body_text[i:i + chunk_size]
                chunk_id = f"{text_id}_chunk{i//chunk_size + 1}"

                new_row = row.copy()
                new_row['Text_ID'] = chunk_id
                new_row['Body'] = chunk
                chunked_rows.append(new_row)

        df = pd.DataFrame(chunked_rows)

    df_meta = pd.concat([df_meta, df], ignore_index=True)

cols = ['Text_ID', 'Title', 'Body', 'Session', 'Sitting', 'Agenda', 'Lang']
df_meta = df_meta.drop(columns=[c for c in cols if c in df_meta.columns], errors='ignore')

print("df_meta shape:", df_meta.shape)
df_meta.head()

df_meta shape: (311354, 17)


,ID,Date,Term,Meeting,Subcorpus,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,Speaker_party_name,Party_status,Party_orientation,Speaker_ID,Speaker_name,Speaker_gender,Speaker_birth,Topic
0,ParlaMint-SI_2013-03-28-SDZ6-Redna-12.u1,2013-03-28,6. mandat,Redna,Reference,Chairperson,MP,notMinister,SD,Social Democrats,Coalition,Centre-left,VeberJanko,"Veber, Janko",M,1960,Domestic Commerce
1,ParlaMint-SI_2013-03-28-SDZ6-Redna-12.u2,2013-03-28,6. mandat,Redna,Reference,Regular,notMP,notMinister,-,-,-,-,KovšcaAlojz,"Kovšca, Alojz",M,-,Domestic Commerce
2,ParlaMint-SI_2013-03-28-SDZ6-Redna-12.u3,2013-03-28,6. mandat,Redna,Reference,Chairperson,MP,notMinister,SD,Social Democrats,Coalition,Centre-left,VeberJanko,"Veber, Janko",M,1960,Macroeconomics
3,ParlaMint-SI_2013-03-28-SDZ6-Redna-12.u4,2013-03-28,6. mandat,Redna,Reference,Regular,MP,notMinister,SD,Social Democrats,Coalition,Centre-left,HanMatjaž,"Han, Matjaž",M,1971,Domestic Commerce
4,ParlaMint-SI_2013-03-28-SDZ6-Redna-12.u5,2013-03-28,6. mandat,Redna,Reference,Chairperson,MP,notMinister,SD,Social Democrats,Coalition,Centre-left,VeberJanko,"Veber, Janko",M,1960,Other


In [7]:
df_sent = pd.DataFrame()
for file in sent_files:
    df = pd.read_csv(file, sep='\t', encoding='utf-8')
    df_sent = pd.concat([df_sent, df])

df_sent = df_sent[df_sent['Element'] == 'u']
cols = ['ID', 'Senti_3', 'Senti_6', 'Senti_n', 'Sents', 'Words', 'Tokens']
df_sent = df_sent[[c for c in cols if c in df_sent.columns]].reset_index(drop=True)
df_sent['ID'] = df_sent['ID'].str.replace(".ana", "", regex=False)

print("df_sent shape: ", df_sent.shape)
df_sent.head()


df_sent shape:  (311354, 7)


,ID,Senti_3,Senti_6,Senti_n,Sents,Words,Tokens
0,ParlaMint-SI_2013-12-19-SDZ6-Redna-20.u1,Neutral,neutral positive,2.9,8,118,136
1,ParlaMint-SI_2013-12-19-SDZ6-Redna-20.u2,Positive,mixed positive,3.67,23,461,533
2,ParlaMint-SI_2013-12-19-SDZ6-Redna-20.u3,Neutral,neutral positive,3.0,3,25,28
3,ParlaMint-SI_2013-12-19-SDZ6-Redna-20.u4,Neutral,neutral negative,1.78,23,514,567
4,ParlaMint-SI_2013-12-19-SDZ6-Redna-20.u5,Neutral,neutral positive,3.0,3,21,26


In [8]:
parla = pd.merge(pd.merge(df_texts, df_meta, on='ID'), df_sent, on='ID')
# Renaming slv metadata to eng (X. mandat -> Term X)
parla['Term'] = parla['Term'].replace({
    '3. mandat': 'Term 3',
    '4. mandat': 'Term 4',
    '5. mandat': 'Term 5',
    '6. mandat': 'Term 6',
    '7. mandat': 'Term 7',
    '8. mandat': 'Term 8'
})

print("df shape: ", parla.shape)
parla.head()


df shape:  (311354, 24)


,ID,Text,Date,Term,Meeting,Subcorpus,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,...,Speaker_name,Speaker_gender,Speaker_birth,Topic,Senti_3,Senti_6,Senti_n,Sents,Words,Tokens
0,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u1,Spoštovane kolegice poslanke in kolegi poslanc...,2013-03-27,Term 6,Redna,Reference,Chairperson,MP,notMinister,SD,...,"Veber, Janko",M,1960,Other,Neutral,neutral positive,3.25,56,986,1149
1,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u2,Hvala lepa za besedo. Spoštovana predsednica V...,2013-03-27,Term 6,Redna,Reference,Regular,MP,notMinister,SLS,...,"Prevc, Mihael",M,1957,Macroeconomics,Negative,negative,0.33,19,284,327
2,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u3,Hvala tudi vam. Dajem besedo predsednici Vlade...,2013-03-27,Term 6,Redna,Reference,Chairperson,MP,notMinister,SD,...,"Veber, Janko",M,1960,Other,Neutral,neutral positive,3.0,3,11,15
3,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u4,"Spoštovane poslanke in poslanci, lep pozdrav p...",2013-03-27,Term 6,Redna,Reference,Regular,MP,notMinister,-,...,"Bratušek, Alenka",F,1970,Macroeconomics,Positive,mixed positive,3.51,29,562,662
4,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u5,Hvala za odgovor. Gospod Prevc bo predstavil z...,2013-03-27,Term 6,Redna,Reference,Chairperson,MP,notMinister,SD,...,"Veber, Janko",M,1960,Other,Neutral,neutral positive,3.0,3,12,15


### Data correction
Due to processing mistake, SDS MP speeches within Term 3 were wrongly labeled as "-" in terms of Party_status (i.e. not Coalition or Opposition). However, SDS was in Opposition for the entire Term 3. 

In [10]:
parla.loc[
    (parla["Speaker_party"] == "SDS") & (parla["Term"] == "Term 3") & (parla["Party_status"] == "-") & (parla["Speaker_MP"] == "MP"),
    "Party_status"
] = "Opposition"

# Now you can filter and inspect as before
sds = parla[parla["Speaker_party"] == "SDS"]
term3 = sds[sds["Term"] == "Term 3"]

print(len(term3))
print(term3["Date"].min())
print(term3["Date"].max())

print(term3["Speaker_MP"].value_counts())

print(term3["Party_status"].value_counts())
print(term3["Speaker_minister"].value_counts())
print(term3["Speaker_MP"].value_counts())

8867
2000-11-10
2004-10-07
Speaker_MP
MP    8867
Name: count, dtype: int64
Party_status
Opposition    8867
Name: count, dtype: int64
Speaker_minister
notMinister    8864
Minister          3
Name: count, dtype: int64
Speaker_MP
MP    8867
Name: count, dtype: int64


## CHES preprocessing
Appending CHES data, mapping numerical values

Party abbreaviations in ParlaMint do not match those in CHES (though adapted for ParlaMint, problem with parties such as ZL and Levica, CHES years coincide with changes within the party structure - either rename or just restructure)

In [11]:
ches = pd.read_csv("../Sample/CHES-SI.tsv", sep="\t", encoding="utf-8")
cols = ["parlamint", "party_id", "year", "lrgen", "galtan", "seat"]
ches = ches[cols]
ches = ches.rename(columns={"parlamint":"Parlamint", "year":"Year", "lrgen":"lrgen", "galtan":"galtan", "seat":"Seat"})
print(ches.shape)
ches.head()

#Parties in ParlaMint, but not in CHES: DL, DLGV, GAS, Konkretno, Lipa, NP, NeP
#Parties in CHES, but not in ParlaMint: AS, SLS-SMS

ches = ches[ches["Parlamint"]!="-"]

ches


(49, 6)


,Parlamint,party_id,Year,lrgen,galtan,Seat
2,DL,-,-,-,-,-
3,DLGV,-,-,-,-,-
4,DeSUS,2906,2002,3.4,5.8,4.4
5,DeSUS,2906,2006,3.2,4.5,4.4
6,DeSUS,2906,2010,4.2,5.2,7.8
7,DeSUS,2906,2014,4.2,5.3,11.1
8,DeSUS,2906,2019,3.8,5.1,5.7
9,GAS,-,-,-,-,-
10,Konkretno,-,-,-,-,-
11,LDS,2901,2002,4.2,3,37.8


In [12]:
#Mapping parties into common (unified) parties column

parties = {
    "ZL": "ZL/Levica",
    "Levica + ZL":"ZL/Levica",
    "Levica": "ZL/Levica",
    "AB":"ZaAB/ZaSLD/SAB",
    "ZaAB":"ZaAB/ZaSLD/SAB",
    "ZaSLD":"ZaAB/ZaSLD/SAB",
    "SAB" :"ZaAB/ZaSLD/SAB",
    "SAB + ZaSLD":"ZaAB/ZaSLD/SAB",
    "ZLSD":"ZLSD/SD",
    "SD":"ZLSD/SD",
    "SD + ZLSD":"ZLSD/SD",
    "SLS+SKD":"SLS+SKD/SLS",
    "SLS":"SLS+SKD/SLS", 
    "DLGV":"DLGV/DL", 
    "DL":"DLGV/DL",
    "SMC":"SMC/GAS/Konkretno",
    "Konkretno": "SMC/GAS/Konkretno"
}
ches["Parties"] = ches['Parlamint'].replace(parties).fillna(ches['Parlamint'])

ches


,Parlamint,party_id,Year,lrgen,galtan,Seat,Parties
2,DL,-,-,-,-,-,DLGV/DL
3,DLGV,-,-,-,-,-,DLGV/DL
4,DeSUS,2906,2002,3.4,5.8,4.4,DeSUS
5,DeSUS,2906,2006,3.2,4.5,4.4,DeSUS
6,DeSUS,2906,2010,4.2,5.2,7.8,DeSUS
7,DeSUS,2906,2014,4.2,5.3,11.1,DeSUS
8,DeSUS,2906,2019,3.8,5.1,5.7,DeSUS
9,GAS,-,-,-,-,-,GAS
10,Konkretno,-,-,-,-,-,SMC/GAS/Konkretno
11,LDS,2901,2002,4.2,3,37.8,LDS


## Merging ParlaMint and CHES dataset into master

In [13]:
## Map the parties on the parliamentary speech data and extract Year from the Date column
ches=ches.drop(columns=["Parlamint"]) #Remove ParlaMint (obsolete) party names from DF
parla["Parties"] = parla['Speaker_party'].replace(parties).fillna(parla['Speaker_party']) 
parla['Year'] = parla['Date'].str.split('-').str[0]


In [14]:
df = parla.merge(
    ches,
    how="left",                     # Keep all speeches
    left_on=["Parties", "Year"],    # ParlaMint side
    right_on=["Parties", "Year"]    # CHES side
)
df.shape


(311354, 30)

In [15]:
cols = ["ID", "Date", "lrgen"]

check = (
    df.loc[df["Parties"] == "ZL/Levica", cols]  
      .sort_values(by="ID", ascending=True)    
      .reset_index(drop=True)                 
)

check.to_csv("Merge_test.csv", index=False)

print(f"Sanity check: {len(check)} rows written to Test.csv") ## Should include lrgen values only for 2014 and 2019
check

Sanity check: 7236 rows written to Test.csv


,ID,Date,lrgen
0,ParlaMint-SI_2014-08-01-SDZ7-Redna-01.u27,2014-08-01,1.0
1,ParlaMint-SI_2014-08-01-SDZ7-Redna-01.u38,2014-08-01,1.0
2,ParlaMint-SI_2014-08-25-SDZ7-Izredna-01.u108,2014-08-25,1.0
3,ParlaMint-SI_2014-08-25-SDZ7-Izredna-01.u11,2014-08-25,1.0
4,ParlaMint-SI_2014-08-25-SDZ7-Izredna-01.u134,2014-08-25,1.0
...,...,...,...
7231,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.u225,2022-04-06,NaN
7232,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.u227,2022-04-06,NaN
7233,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.u229,2022-04-06,NaN
7234,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.u76,2022-04-06,NaN


In [16]:
df.sort_values(by="ID", ascending=True)
df["Senti_n"] = pd.to_numeric(df["Senti_n"], errors="coerce")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311354 entries, 0 to 311353
Data columns (total 30 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ID                  311354 non-null  object 
 1   Text                311354 non-null  object 
 2   Date                311354 non-null  object 
 3   Term                311354 non-null  object 
 4   Meeting             311354 non-null  object 
 5   Subcorpus           311354 non-null  object 
 6   Speaker_role        311354 non-null  object 
 7   Speaker_MP          311354 non-null  object 
 8   Speaker_minister    311354 non-null  object 
 9   Speaker_party       311354 non-null  object 
 10  Speaker_party_name  311354 non-null  object 
 11  Party_status        311354 non-null  object 
 12  Party_orientation   311354 non-null  object 
 13  Speaker_ID          311354 non-null  object 
 14  Speaker_name        311354 non-null  object 
 15  Speaker_gender      311354 non-nul

In [ ]:
#Sample dataset saved as CSV/TSV
#df.to_csv("../Sample/Datasets/Sample_master.tsv", sep='\t', encoding='utf-8', index=False)

In [21]:
#Full dataset saved as TSV
df.to_csv("../ParlaMint-SI_full_dataset.tsv", sep='\t', encoding='utf-8', index=False)